In [1]:
# 1. Clone only the repository metadata
!git clone --depth 1 --filter=blob:none --sparse https://github.com/EconShark/23_big_data_analysis_class.git

# 2. Check out only the Baitap03/data directory
!cd 23_big_data_analysis_class && git sparse-checkout set Baitap03/data

# 3. Copy the datasets to /content/data
!mkdir -p /content/data
!cp -rn 23_big_data_analysis_class/Baitap03/data/. /content/data/

# 4. Remove the cloned repository
!rm -rf 23_big_data_analysis_class

Cloning into '23_big_data_analysis_class'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 30 (delta 0), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 6.35 KiB | 6.35 MiB/s, done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 0), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 168.31 KiB | 1.98 MiB/s, done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 3.57 MiB | 8.13 MiB/s, done.
Updating files: 100% (4/4), done.
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead


<a id="setup"></a>
## Khởi tạo SparkSession

In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    print("Đang cài PySpark...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
else:
    print("PySpark đã sẵn sàng.")

PySpark đã sẵn sàng.


In [3]:
from pathlib import Path

candidates = [
    Path.cwd() / "data",
    Path("/content/data"),
    Path.cwd().parent / "data"
]
DATA_DIR = next((p for p in candidates if p.exists()), None)

print("DATA_DIR =", DATA_DIR)
if DATA_DIR:
    for p in sorted(DATA_DIR.iterdir()):
        print(" -", p.name)

DATA_DIR = /content/data
 - flights-larger.csv
 - sms.csv


In [4]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("MachineLearningWithPySpark")
    .getOrCreate()
)

print("PySpark version:", pyspark.__version__)
print("Spark version  :", spark.version)
print("Master         :", spark.sparkContext.master)

PySpark version: 4.0.4
Spark version  : 4.0.4
Master         : local[*]


<a id="loading"></a>
## 1. Đọc và khảo sát dữ liệu

- `flights-larger.csv`: bài toán dự đoán chuyến bay trễ và thời gian bay.
- `sms.csv`: bài toán phân loại SMS spam.

### 1.1. Dữ liệu chuyến bay

In [5]:
flights = spark.read.csv(
    str(DATA_DIR / "flights-larger.csv"),
    header=True,
    inferSchema=True,
    nullValue="NA"
)

flights.printSchema()
flights.show(5)
print("Số bản ghi:", flights.count())

root
 |-- mon: integer (nullable = true)
 |-- dom: integer (nullable = true)
 |-- dow: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- org: string (nullable = true)
 |-- mile: integer (nullable = true)
 |-- depart: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- delay: integer (nullable = true)

+---+---+---+-------+------+---+----+------+--------+-----+
|mon|dom|dow|carrier|flight|org|mile|depart|duration|delay|
+---+---+---+-------+------+---+----+------+--------+-----+
| 10| 10|  1|     OO|  5836|ORD| 157|  8.18|      51|   27|
|  1|  4|  1|     OO|  5866|ORD| 466|  15.5|     102| NULL|
| 11| 22|  1|     OO|  6016|ORD| 738|  7.17|     127|  -19|
|  2| 14|  5|     B6|   199|JFK|2248| 21.17|     365|   60|
|  5| 25|  3|     WN|  1675|SJC| 386| 12.92|      85|   22|
+---+---+---+-------+------+---+----+------+--------+-----+
only showing top 5 rows
Số bản ghi: 275000


### 1.2. Các cột chính

- `mon`: tháng
- `dom`: ngày trong tháng
- `dow`: ngày trong tuần
- `carrier`: hãng bay
- `flight`: số hiệu chuyến bay
- `org`: sân bay xuất phát
- `mile`: quãng đường
- `depart`: giờ khởi hành
- `duration`: thời gian bay
- `delay`: độ trễ

### 1.3. Dữ liệu SMS với schema tường minh

Định nghĩa `StructType`.

In [7]:
sms_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("label", IntegerType(), True)
])

sms = spark.read.csv(
    str(DATA_DIR / "sms.csv"),
    sep=";",
    header=False,
    schema=sms_schema
)

sms.show(5, truncate=False)
sms.printSchema()

+---+---------------------------------------------------------------------------------------------------------------+-----+
|id |text                                                                                                           |label|
+---+---------------------------------------------------------------------------------------------------------------+-----+
|1  |Sorry, I'll call later in meeting                                                                              |0    |
|2  |Dont worry. I guess he's busy.                                                                                 |0    |
|3  |Call FREEPHONE 0800 542 0578 now!                                                                              |1    |
|4  |Win a 1000 cash prize or a prize worth 5000                                                                    |1    |
|5  |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...|0    |
+---+---

<a id="prep"></a>
## 2. Chuẩn bị dữ liệu cho Machine Learning

Trong Machine Learning, dữ liệu thô thường chưa thể đưa trực tiếp vào mô hình.

Các bước thường gặp:

1. loại cột không cần thiết;
2. xử lý missing values;
3. chuyển đổi đơn vị;
4. tạo label;
5. mã hóa biến phân loại;
6. ghép các predictor thành `features`.

### 2.1. Kiểm tra missing values

In [8]:
flights.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in flights.columns
]).show()

+---+---+---+-------+------+---+----+------+--------+-----+
|mon|dom|dow|carrier|flight|org|mile|depart|duration|delay|
+---+---+---+-------+------+---+----+------+--------+-----+
|  0|  0|  0|      0|     0|  0|   0|     0|       0|16711|
+---+---+---+-------+------+---+----+------+--------+-----+



### 2.2. Làm sạch và tạo label
`withColumn()` được dùng để tạo feature mới.

Ở đây:

- mile → km;
- tạo `label = 1` nếu delay >= 15 phút.

In [10]:
flights_ml = (
    flights
    .drop("flight")
    .filter(F.col("delay").isNotNull())
    .dropna()
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
    .drop("mile")
    .withColumn("label", (F.col("delay") >= 15).cast("integer"))
)

flights_ml.select(
    "mon", "carrier", "org", "km", "duration", "delay", "label"
).show(10)

+---+-------+---+------+--------+-----+-----+
|mon|carrier|org|    km|duration|delay|label|
+---+-------+---+------+--------+-----+-----+
| 10|     OO|ORD| 253.0|      51|   27|    1|
| 11|     OO|ORD|1188.0|     127|  -19|    0|
|  2|     B6|JFK|3618.0|     365|   60|    1|
|  5|     WN|SJC| 621.0|      85|   22|    1|
|  3|     B6|LGA|1732.0|     182|   70|    1|
|  5|     B6|ORD|1191.0|     130|   47|    1|
|  1|     UA|SFO|1093.0|     123|  135|    1|
|  8|     US|LGA| 344.0|      71|  -10|    0|
|  5|     AA|ORD|1926.0|     195|  -11|    0|
|  8|     B6|JFK|1902.0|     198|   20|    1|
+---+-------+---+------+--------+-----+-----+
only showing top 10 rows


<a id="categorical"></a>
## 3. StringIndexer, OneHotEncoder và VectorAssembler

Machine Learning algorithms thường cần dữ liệu số.

### 3.1. StringIndexer

`StringIndexer` biến chuỗi category thành chỉ số số.

Ví dụ:

```text
AA -> 0
OO -> 1
B6 -> 2
...
```

Thứ tự mặc định thường phụ thuộc tần suất xuất hiện.

In [14]:
from pyspark.ml.feature import StringIndexer

carrier_indexer = StringIndexer(
    inputCol="carrier",
    outputCol="carrier_idx",
    handleInvalid="keep"
)

flights_idx = carrier_indexer.fit(flights_ml).transform(flights_ml)

# Khởi tạo StringIndexer cho cột 'org'
org_indexer = StringIndexer(
    inputCol="org",
    outputCol="org_idx",
    handleInvalid="keep"
)

# Khớp và chuyển đổi trên flights_idx để thêm cột org_idx
flights_idx = org_indexer.fit(flights_idx).transform(flights_idx)

### 3.2. OneHotEncoder

One-Hot Encoding tạo vector nhị phân/sparse vector, tránh tạo quan hệ thứ tự giả giữa các category.

In [12]:
from pyspark.ml.feature import OneHotEncoder

onehot = OneHotEncoder(
    inputCols=["org_idx"],
    outputCols=["org_dummy"],
    dropLast=True
)

<a id="assembler"></a>
### 3.3. VectorAssembler

Spark ML yêu cầu predictor được gom trong một cột vector tên thường là **`features`**.

`VectorAssembler` thực hiện việc này.

In [15]:
from pyspark.ml.feature import VectorAssembler

# Khởi tạo VectorAssembler để gộp các đặc trưng
assembler = VectorAssembler(
    inputCols=[
        "mon", "dom", "dow",
        "carrier_idx", "org_idx",
        "km", "depart", "duration"
    ],
    outputCol="features"
)

# Chuyển đổi dữ liệu sẵn sàng cho mô hình
flights_ready = assembler.transform(flights_idx)

flights_ready.select(
    "features", "label", "delay"
).show(5, truncate=False)

+-----------------------------------------+-----+-----+
|features                                 |label|delay|
+-----------------------------------------+-----+-----+
|[10.0,10.0,1.0,2.0,0.0,253.0,8.18,51.0]  |1    |27   |
|[11.0,22.0,1.0,2.0,0.0,1188.0,7.17,127.0]|0    |-19  |
|[2.0,14.0,5.0,4.0,2.0,3618.0,21.17,365.0]|1    |60   |
|[5.0,25.0,3.0,3.0,5.0,621.0,12.92,85.0]  |1    |22   |
|[3.0,28.0,1.0,4.0,3.0,1732.0,13.33,182.0]|1    |70   |
+-----------------------------------------+-----+-----+
only showing top 5 rows


### 3.4. Chia train/test

In [16]:
train_df, test_df = flights_ready.randomSplit(
    [0.8, 0.2],
    seed=23
)

print("Train:", train_df.count())
print("Test :", test_df.count())
print("Train ratio:", train_df.count() / flights_ready.count())

Train: 206212
Test : 52077
Train ratio: 0.7983770117968632


<a id="dt"></a>
## 4. Decision Tree Classification

Decision Tree phân chia dữ liệu qua các nút:

```text
Root
 ├── nhánh trái
 └── nhánh phải
```

Quá trình tiếp tục cho đến khi tạo các leaf node.

Decision Tree được dùng để phân loại xe theo nơi sản xuất.  
Ở phần mở rộng, ta dùng nó để dự đoán chuyến bay có trễ >= 15 phút hay không.

In [17]:
from pyspark.ml.classification import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    seed=23
)

tree_model = tree.fit(train_df)
tree_pred = tree_model.transform(test_df)

tree_pred.select(
    "label", "prediction", "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |0.0       |[0.6225186017127615,0.37748139828723853]|
|1    |1.0       |[0.3195589266923886,0.6804410733076114] |
|0    |1.0       |[0.3195589266923886,0.6804410733076114] |
|1    |0.0       |[0.6225186017127615,0.37748139828723853]|
|1    |1.0       |[0.3195589266923886,0.6804410733076114] |
|0    |1.0       |[0.3195589266923886,0.6804410733076114] |
|1    |1.0       |[0.3195589266923886,0.6804410733076114] |
|1    |0.0       |[0.6225186017127615,0.37748139828723853]|
|0    |0.0       |[0.6225186017127615,0.37748139828723853]|
|0    |0.0       |[0.6225186017127615,0.37748139828723853]|
+-----+----------+----------------------------------------+
only showing top 10 rows


### 4.1. Xem cấu trúc cây

In [18]:
print(tree_model.toDebugString[:5000])

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_b203aef5b506, depth=5, numNodes=43, numClasses=2, numFeatures=8
  If (feature 6 <= 10.96)
   If (feature 4 in {4.0,5.0,6.0,7.0})
    If (feature 7 <= 251.5)
     Predict: 0.0
    Else (feature 7 > 251.5)
     If (feature 3 in {1.0,4.0,6.0})
      If (feature 4 in {5.0})
       Predict: 0.0
      Else (feature 4 not in {5.0})
       Predict: 1.0
     Else (feature 3 not in {1.0,4.0,6.0})
      If (feature 0 <= 8.5)
       Predict: 0.0
      Else (feature 0 > 8.5)
       Predict: 1.0
   Else (feature 4 not in {4.0,5.0,6.0,7.0})
    If (feature 6 <= 9.49)
     Predict: 0.0
    Else (feature 6 > 9.49)
     If (feature 0 <= 5.5)
      If (feature 5 <= 1050.5)
       Predict: 0.0
      Else (feature 5 > 1050.5)
       Predict: 1.0
     Else (feature 0 > 5.5)
      If (feature 0 <= 10.5)
       Predict: 0.0
      Else (feature 0 > 10.5)
       Predict: 1.0
  Else (feature 6 > 10.96)
   If (feature 4 in {4.0,5.0,6.0,7.0})
    If (feat

<a id="logistic"></a>
## 5. Logistic Regression và đánh giá classifier

Logistic Regression dùng hàm logistic để biến linear score thành xác suất trong khoảng 0–1.

Mặc định:

- xác suất >= ngưỡng → class 1;
- xác suất < ngưỡng → class 0.

PySpark:

In [19]:
from pyspark.ml.classification import LogisticRegression

logistic = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=50
)

log_model = logistic.fit(train_df)
log_pred = log_model.transform(test_df)

log_pred.select(
    "label", "prediction", "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |0.0       |[0.557245791644765,0.44275420835523505] |
|1    |1.0       |[0.2646308554040308,0.7353691445959691] |
|0    |1.0       |[0.2438573328127501,0.7561426671872499] |
|1    |0.0       |[0.5405863866128278,0.45941361338717224]|
|1    |1.0       |[0.29158128138704686,0.7084187186129531]|
|0    |1.0       |[0.31373727084730363,0.6862627291526964]|
|1    |1.0       |[0.3180490430367354,0.6819509569632646] |
|1    |1.0       |[0.4241927240351956,0.5758072759648044] |
|0    |0.0       |[0.5356616493670768,0.46433835063292317]|
|0    |1.0       |[0.49707685136788693,0.5029231486321131]|
+-----+----------+----------------------------------------+
only showing top 10 rows


<a id="class_eval"></a>
### 5.1. Confusion Matrix

Ta cần bốn thành phần:

- TP: dự đoán 1, thực tế 1.
- FP: dự đoán 1, thực tế 0.
- FN: dự đoán 0, thực tế 1.
- TN: dự đoán 0, thực tế 0.

In [21]:
log_pred.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0| 9675|
|    0|       0.0|14754|
|    1|       1.0|16798|
|    0|       1.0|10850|
+-----+----------+-----+



### 5.2. Accuracy, Precision, Recall, F1, AUC

In [22]:
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

multi_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

binary_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction"
)

for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    value = multi_eval.evaluate(
        log_pred,
        {multi_eval.metricName: metric}
    )
    print(f"{metric:18s}: {value:.4f}")

auc = binary_eval.evaluate(log_pred)
print(f"{'AUC':18s}: {auc:.4f}")

accuracy          : 0.6059
f1                : 0.6055
weightedPrecision : 0.6058
weightedRecall    : 0.6059
AUC               : 0.6456


<a id="text"></a>
## 6. Text Mining và SMS Spam Classification

Slide trình bày pipeline:

```text
Text
 ↓
Remove punctuation
 ↓
Tokenizer
 ↓
StopWordsRemover
 ↓
HashingTF
 ↓
IDF
 ↓
features
```

Đây là cách biến văn bản thành vector số cho Machine Learning.

### 6.1. Làm sạch và tokenize

In [26]:
from pyspark.sql.functions import regexp_replace

sms_clean = sms.withColumn(
    "text",
    regexp_replace("text", r"[_():;,.!?\-]", " ")
)

sms_clean = sms_clean.withColumn(
    "text",
    regexp_replace("text", r"[0-9]", " ")
)

sms_clean = sms_clean.withColumn(
    "text",
    regexp_replace("text", r" +", " ")
)

sms_clean.show(5, truncate=False)

+---+-------------------------------------------------------------------------------------------------------+-----+
|id |text                                                                                                   |label|
+---+-------------------------------------------------------------------------------------------------------+-----+
|1  |Sorry I'll call later in meeting                                                                       |0    |
|2  |Dont worry I guess he's busy                                                                           |0    |
|3  |Call FREEPHONE now                                                                                     |1    |
|4  |Win a cash prize or a prize worth                                                                      |1    |
|5  |Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat |0    |
+---+-------------------------------------------------------------------

#### Tokenizer

In [27]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="text",
    outputCol="tokens"
)

sms_tokens = tokenizer.transform(sms_clean)

sms_tokens.select("text", "tokens").show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+
|text                                                                                                   |tokens                                                                                                                     |
+-------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+
|Sorry I'll call later in meeting                                                                       |[sorry, i'll, call, later, in, meeting]                                                                                    |
|Dont worry I guess he's busy                                                   

#### StopWordsRemover

In [28]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="words"
)

sms_words = remover.transform(sms_tokens)

sms_words.select("tokens", "words").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|tokens                                                                                                                     |words                                                                                              |
+---------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|[sorry, i'll, call, later, in, meeting]                                                                                    |[sorry, call, later, meeting]                                                                      |
|[dont, worry, i, guess, he's, busy]                                                            

### 6.2. HashingTF + IDF

In [29]:
from pyspark.ml.feature import HashingTF

hasher = HashingTF(
    inputCol="words",
    outputCol="hash",
    numFeatures=1024
)

sms_hash = hasher.transform(sms_words)

sms_hash.select("id", "words", "hash").show(5, truncate=False)

+---+---------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
|id |words                                                                                              |hash                                                                                                                                     |
+---+---------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
|1  |[sorry, call, later, meeting]                                                                      |(1024,[138,384,577,996],[1.0,1.0,1.0,1.0])                                                                                               |
|2  |[dont, worry, guess

In [30]:
from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="hash",
    outputCol="features"
)

idf_model = idf.fit(sms_hash)
sms_tfidf = idf_model.transform(sms_hash)

sms_tfidf.select(
    "id", "label", "features"
).show(5, truncate=False)

+---+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id |label|features                                                                                                                                                                                                                                                                                                                                                                        |
+---+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

<a id="sms"></a>
### 6.3. Logistic Regression cho SMS

Bài toán:

```text
Input : SMS text
Output: 0 = normal, 1 = spam
```

Ta dùng Logistic Regression trên TF-IDF features.

In [31]:
sms_model_df = sms_tfidf.select("label", "features")

sms_train, sms_test = sms_model_df.randomSplit(
    [0.8, 0.2],
    seed=13
)

sms_logistic = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    regParam=0.2
)

sms_log_model = sms_logistic.fit(sms_train)
sms_pred = sms_log_model.transform(sms_test)

sms_pred.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0|   39|
|    0|       0.0|  932|
|    1|       1.0|  121|
|    0|       1.0|    4|
+-----+----------+-----+



In [32]:
sms_auc = BinaryClassificationEvaluator(
    labelCol="label"
).evaluate(sms_pred)

sms_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    metricName="f1"
).evaluate(sms_pred)

print("SMS AUC:", sms_auc)
print("SMS F1 :", sms_f1)

SMS AUC: 0.9912793803418793
SMS F1 : 0.9587173795339076


<a id="regression"></a>
## 7. Linear Regression

Linear Regression để dự đoán fuel consumption.  
Phần mở rộng dùng dataset chuyến bay để dự đoán **duration**.

Ta thử mô hình đơn giản:

```text
duration ~ km
```

In [33]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

reg_base = (
    flights
    .filter(F.col("duration").isNotNull())
    .filter(F.col("mile").isNotNull())
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
)

reg_assembler = VectorAssembler(
    inputCols=["km"],
    outputCol="features"
)

reg_data = reg_assembler.transform(reg_base)

reg_train, reg_test = reg_data.randomSplit([0.8, 0.2], seed=42)

In [34]:
lr_model = lr = LinearRegression(
    labelCol="duration",
    featuresCol="features"
).fit(reg_train)


reg_pred = lr_model.transform(reg_test)

reg_pred.select(
    "km", "duration", "prediction"
).show(10)

+------+--------+------------------+
|    km|duration|        prediction|
+------+--------+------------------+
|4001.0|     300| 347.3336415989867|
|3983.0|     375| 345.9700268069072|
|1395.0|     135|149.91252225681146|
|1926.0|     180|190.13915862315645|
|1180.0|     115| 133.6249011291953|
|1180.0|     130| 133.6249011291953|
|1180.0|     130| 133.6249011291953|
|1091.0|     110| 126.8825835461356|
|2356.0|     205|222.71440087838872|
|2659.0|     251|245.66858321172683|
+------+--------+------------------+
only showing top 10 rows


### 7.1. Các metric regression

In [37]:
for metric in ["rmse", "mae", "mse", "r2"]:
    evaluator = RegressionEvaluator(
        labelCol="duration",
        predictionCol="prediction",
        metricName=metric
    )
    print(f"{metric:4s}: {evaluator.evaluate(reg_pred):.4f}")

rmse: 17.1476
mae : 13.1121
mse : 294.0405
r2  : 0.9613


### 7.2. Intercept và coefficient

In [38]:
print("Intercept   :", lr_model.intercept)
print("Coefficients:", lr_model.coefficients)

Intercept   : 44.23237587065089
Coefficients: [0.07575637733774951]


<a id="engineering"></a>
## 8. Feature Engineering và Bucketing

- bucketing;
- log, sqrt, power;
- product;
- ratio.

Bucketing biến biến liên tục thành nhóm rời rạc.

### 8.1. Bucketizer cho giờ khởi hành

In [39]:
from pyspark.ml.feature import Bucketizer

# Dữ liệu depart có dạng giờ thập phân khoảng 0–24
bucketizer = Bucketizer(
    splits=[0, 6, 12, 18, 24],
    inputCol="depart",
    outputCol="depart_bucket",
    handleInvalid="keep"
)

bucketed = bucketizer.transform(
    flights.filter(F.col("depart").isNotNull())
)

bucketed.select(
    "depart", "depart_bucket"
).show(10)

bucketed.groupBy("depart_bucket").count().orderBy("depart_bucket").show()

+------+-------------+
|depart|depart_bucket|
+------+-------------+
|  8.18|          1.0|
|  15.5|          2.0|
|  7.17|          1.0|
| 21.17|          3.0|
| 12.92|          2.0|
| 13.33|          2.0|
|  9.58|          1.0|
| 12.75|          2.0|
|  13.0|          2.0|
| 14.42|          2.0|
+------+-------------+
only showing top 10 rows
+-------------+------+
|depart_bucket| count|
+-------------+------+
|          0.0|   911|
|          1.0| 98550|
|          2.0|104471|
|          3.0| 71068|
+-------------+------+



### 8.2. Feature công thức

In [40]:
engineered = (
    flights
    .filter(F.col("mile").isNotNull())
    .filter(F.col("duration").isNotNull())
    .withColumn("km", F.col("mile") * 1.60934)
    .withColumn("speed_km_per_min", F.col("km") / F.col("duration"))
    .withColumn("sqrt_km", F.sqrt("km"))
    .withColumn("log_km", F.log1p("km"))
)

engineered.select(
    "km", "duration", "speed_km_per_min", "sqrt_km", "log_km"
).show(10)

+----------+--------+-----------------+------------------+------------------+
|        km|duration| speed_km_per_min|           sqrt_km|            log_km|
+----------+--------+-----------------+------------------+------------------+
| 252.66638|      51|4.954242745098039| 15.89548300618764| 5.536019939100953|
| 749.95244|     102|7.352474901960784|27.385259538664226| 6.621342320863969|
|1187.69292|     127|9.351912755905511| 34.46292094411035|  7.08060959588236|
|3617.79632|     365|9.911770739726027| 60.14811983761421| 8.193896741239387|
| 621.20524|      85| 7.30829694117647|24.923989247309507|6.4332700061574215|
|1731.64984|     182| 9.51455956043956| 41.61309697679326| 7.457407215058698|
| 1190.9116|     130| 9.16085846153846|34.509587073739375| 7.083313683801401|
|1092.74186|     123|8.884080162601627| 33.05664623037249|6.9973599953584165|
| 344.39876|      71| 4.85068676056338|18.557983726687553| 5.844699575665601|
|1926.37998|     195|9.878871692307692| 43.89054545115611|  7.56

<a id="regularization"></a>
## 9. Regularization: Ridge, Lasso, Elastic Net

Linear Regression chuẩn tối thiểu hóa lỗi.  
Regularization bổ sung penalty lên coefficient.

### Ridge

- `elasticNetParam = 0`
- penalty L2.

### Lasso

- `elasticNetParam = 1`
- penalty L1.
- có thể đẩy một số coefficient về 0.

### Elastic Net

- `0 < elasticNetParam < 1`
- kết hợp L1 và L2.

In [42]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

# Tái tạo lại DataFrame 'reg_full' bị thiếu do ô trước bị xóa
reg_idx = StringIndexer(
    inputCol="org",
    outputCol="org_idx",
    handleInvalid="keep"
).fit(reg_base).transform(reg_base)

reg_ohe = OneHotEncoder(
    inputCols=["org_idx"],
    outputCols=["org_dummy"]
).fit(reg_idx).transform(reg_idx)

reg_assembler2 = VectorAssembler(
    inputCols=["km", "org_dummy"],
    outputCol="features2"
)

reg_full = reg_assembler2.transform(reg_ohe)

# Thực hiện chia tập dữ liệu train/test
reg_train3, reg_test3 = reg_full.randomSplit([0.8, 0.2], seed=23)

models_reg = []

for name, alpha in [
    ("Ridge", 0.0),
    ("ElasticNet", 0.5),
    ("Lasso", 1.0)
]:
    model = LinearRegression(
        labelCol="duration",
        featuresCol="features2",
        regParam=0.1,
        elasticNetParam=alpha
    ).fit(reg_train3)

    pred = model.transform(reg_test3)
    rmse = RegressionEvaluator(
        labelCol="duration",
        predictionCol="prediction",
        metricName="rmse"
    ).evaluate(pred)

    models_reg.append({
        "model": name,
        "rmse": rmse,
        "num_coefficients": len(model.coefficients),
        "zero_coefficients": sum(abs(x) < 1e-10 for x in model.coefficients)
    })

pd.DataFrame(models_reg)

,model,rmse,num_coefficients,zero_coefficients
0,Ridge,11.289664,9,0
1,ElasticNet,11.291365,9,1
2,Lasso,11.294537,9,0


<a id="pipeline"></a>
## 10. Pipeline và Data Leakage

```text
.fit()       -> chỉ dùng TRAIN
.transform() -> dùng cho TRAIN và TEST
```

Nếu fit bộ tiền xử lý trên toàn bộ dataset trước khi chia train/test, test data đã tác động vào preprocessing → **data leakage**.

`Pipeline` giúp đóng gói các stage đúng thứ tự.

### Pipeline Regression (Minh họa)

In [43]:
from pyspark.ml import Pipeline

pipeline_data = (
    flights
    .filter(F.col("duration").isNotNull())
    .filter(F.col("mile").isNotNull())
    .filter(F.col("org").isNotNull())
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
)

pipe_train, pipe_test = pipeline_data.randomSplit(
    [0.8, 0.2],
    seed=13
)

pipe_indexer = StringIndexer(
    inputCol="org",
    outputCol="org_idx",
    handleInvalid="keep"
)

pipe_onehot = OneHotEncoder(
    inputCols=["org_idx"],
    outputCols=["org_dummy"]
)

pipe_assembler = VectorAssembler(
    inputCols=["km", "org_dummy"],
    outputCol="features"
)

pipe_lr = LinearRegression(
    labelCol="duration",
    featuresCol="features"
)

pipeline = Pipeline(
    stages=[
        pipe_indexer,
        pipe_onehot,
        pipe_assembler,
        pipe_lr
    ]
)

pipeline_model = pipeline.fit(pipe_train)
pipe_pred = pipeline_model.transform(pipe_test)

pipe_rmse = RegressionEvaluator(
    labelCol="duration",
    metricName="rmse"
).evaluate(pipe_pred)

print("Pipeline RMSE:", pipe_rmse)

Pipeline RMSE: 11.187971350851406


<a id="cv"></a>
## 11. Cross Validation và Grid Search

Train/test split chỉ cho một phép chia dữ liệu.

Cross Validation:

1. chia training data thành nhiều fold;
2. lần lượt dùng mỗi fold làm validation;
3. tính metric trung bình;
4. chọn model/hyperparameter tốt nhất.

Test set vẫn được giữ riêng để đánh giá cuối cùng.

In [45]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

cv_regression = LinearRegression(
    labelCol="duration",
    featuresCol="features"
)

cv_pipeline = Pipeline(
    stages=[
        pipe_indexer,
        pipe_onehot,
        pipe_assembler,
        cv_regression
    ]
)

param_grid = (
    ParamGridBuilder()
    .addGrid(cv_regression.regParam, [0.01, 0.1, 1.0])
    .addGrid(cv_regression.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

print("Số cấu hình:", len(param_grid))

cv_evaluator = RegressionEvaluator(
    labelCol="duration",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=cv_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=3,
    seed=13,
    parallelism=2
)

Số cấu hình: 9


In [46]:
# Huấn luyện cross-validation
# Có thể mất vài phút tùy máy.
cv_model = cv.fit(pipe_train)

best_predictions = cv_model.transform(pipe_test)
best_rmse = cv_evaluator.evaluate(best_predictions)

print("Best test RMSE:", best_rmse)
print("Average CV metrics:")
print(cv_model.avgMetrics)

Best test RMSE: 11.18794629815799
Average CV metrics:
[np.float64(11.21222312526637), np.float64(11.212220771736682), np.float64(11.212250742825184), np.float64(11.212645439450151), np.float64(11.213404714714505), np.float64(11.215621374066805), np.float64(11.253953198119447), np.float64(11.329378009751911), np.float64(11.546458968429102)]


### 11.1. Xem mô hình tốt nhất

In [47]:
best_pipeline = cv_model.bestModel
best_lr = best_pipeline.stages[-1]

print("regParam       =", best_lr.getRegParam())
print("elasticNetParam=", best_lr.getElasticNetParam())

regParam       = 0.01
elasticNetParam= 0.5


<a id="ensemble"></a>
## 12. Ensemble Models

**Ensemble**: kết hợp nhiều cây/mô hình.

Các ví dụ trong code ZIP:

- Decision Tree;
- Gradient-Boosted Trees (GBT);
- Random Forest;
- Cross Validation cho Random Forest.

In [49]:
ensemble_data = (
    flights
    .filter(F.col("delay").isNotNull())
    .filter(F.col("duration").isNotNull())
    .filter(F.col("depart").isNotNull())
    .filter(F.col("mon").isNotNull())
    .withColumn("label", (F.col("delay") >= 15).cast("integer"))
)

ensemble_assembler = VectorAssembler(
    inputCols=["mon", "depart", "duration"],
    outputCol="features"
)

ensemble_data = ensemble_assembler.transform(ensemble_data)

ens_train, ens_test = ensemble_data.randomSplit(
    [0.8, 0.2],
    seed=17
)

from pyspark.ml.classification import (
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

models = {
    "Decision Tree": DecisionTreeClassifier(seed=17),
    "Random Forest": RandomForestClassifier(numTrees=30, seed=17),
    "GBT": GBTClassifier(maxIter=20, seed=17)
}

### 12.1. So sánh mô hình (Decision Tree, RF, GBT)

In [50]:
ensemble_results = []

for name, estimator in models.items():
    model = estimator.fit(ens_train)
    pred = model.transform(ens_test)

    auc = BinaryClassificationEvaluator(
        labelCol="label"
    ).evaluate(pred)

    f1 = MulticlassClassificationEvaluator(
        labelCol="label",
        metricName="f1"
    ).evaluate(pred)

    ensemble_results.append({
        "model": name,
        "AUC": auc,
        "F1": f1
    })

pd.DataFrame(ensemble_results).sort_values("AUC", ascending=False)

,model,AUC,F1
2,GBT,0.679002,0.628652
1,Random Forest,0.666685,0.620376
0,Decision Tree,0.618192,0.620846


<a id="project"></a>
## 4. BÀI TẬP MỞ RỘNG

### Project A - Flight Delay Classification

#### Input

- month, day-of-month, day-of-week;
- carrier;
- origin airport;
- distance;
- departure time;
- duration.

#### Output

```text
label = 1 nếu delay >= 15 phút
label = 0 nếu ngược lại
```

#### Yêu cầu

1. Làm sạch dữ liệu.
2. Tạo label: delay >= 15 phút.
3. Mã hóa carrier và org.
4. Tạo Pipeline preprocessing.
5. So sánh:
   - Logistic Regression;
   - Decision Tree;
   - Random Forest;
   - GBT.
5. Báo cáo Accuracy, F1, AUC.
6. Tuning ít nhất một mô hình bằng Cross Validation.
7. Giải thích vì sao chọn mô hình cuối cùng.

### Project B - SMS Spam Classification

#### Input

Nội dung SMS.

#### Output

- 0: normal;
- 1: spam.

#### Pipeline

```text
Text
 → Tokenizer
 → StopWordsRemover
 → HashingTF
 → IDF
 → LogisticRegression
```

1. Làm sạch text và tokenize.
2. Loại stop words.
3. HashingTF + IDF.
4. Logistic Regression.
5. Thử numFeatures = 1024, 4096, 16384.
6. Thử regParam/elasticNetParam.
7. So sánh F1 và AUC.

<a id="exercise"></a>
## 5. BÀI TẬP THÊM

### Bài 1 - Data Preparation

Với `flights-larger.csv`:

1. đếm missing value ở từng cột;
2. loại `flight`;
3. loại dòng thiếu `delay`;
4. tạo `km`;
5. tạo `label`.

---

### Bài 2 - StringIndexer và OneHotEncoder

1. index `carrier`;
2. index `org`;
3. one-hot encode `org`;
4. giải thích tại sao không nên dùng index trực tiếp như một biến liên tục.

---

### Bài 3 - Decision Tree

1. huấn luyện cây với `maxDepth = 3, 5, 10`;
2. so sánh Accuracy/F1/AUC;
3. in `toDebugString`.

---

### Bài 4 - Logistic Regression

1. huấn luyện Logistic Regression;
2. tính confusion matrix;
3. tính Precision, Recall, F1;
4. tính AUC.

---

### Bài 5 - SMS

1. loại punctuation;
2. tokenize;
3. remove stop words;
4. HashingTF;
5. IDF;
6. Logistic Regression;
7. tính F1 và AUC.

---

### Bài 6 - Regression

Dự đoán `duration`.

1. model chỉ dùng `km`;
2. model dùng `km + org`;
3. so sánh RMSE;
4. giải thích coefficient.

---

### Bài 7 - Regularization

Huấn luyện:

- Linear Regression;
- Ridge;
- Lasso;
- Elastic Net.

So sánh:

- RMSE;
- số coefficient gần 0.

---

### Bài 8 - Pipeline

Xây dựng Pipeline:

```text
StringIndexer
 → OneHotEncoder
 → VectorAssembler
 → LinearRegression
```

Giải thích tại sao phải fit pipeline trên training data.

---

### Bài 9 - Cross Validation

Grid:

```text
regParam = [0.01, 0.1, 1.0]
elasticNetParam = [0, 0.5, 1]
```

1. có bao nhiêu model configuration?
2. chạy 3-fold CV;
3. tìm model tốt nhất;
4. đánh giá trên test set.

---

### Bài 10 - Ensemble

So sánh:

- Decision Tree;
- Random Forest;
- GBT.

Chọn mô hình dựa trên AUC và F1.

## 6. Câu hỏi ôn tập và yêu cầu nộp bài

Câu hỏi ôn tập

1. Regression và Classification khác nhau thế nào?
2. StringIndexer khác OneHotEncoder ở điểm nào?
3. Tại sao Spark ML cần cột features dạng vector?
4. Confusion matrix gồm những thành phần nào?
5. Khi nào Accuracy có thể gây hiểu nhầm?
6. TF-IDF giúp biểu diễn văn bản như thế nào?
7. RMSE và R² có ý nghĩa gì?
8. Lasso khác Ridge thế nào?
9. Data leakage xảy ra khi nào?
10. Pipeline hỗ trợ tránh leakage như thế nào?
11. Cross Validation dùng training set hay test set?
12. Random Forest khác Decision Tree đơn như thế nào?